# EndoVLM Zero-Shot Inference Example

Zero-shot upper-GI anatomy classification with prompt-based recognition:
each bundled sample image (`data/zeroshot/`, hyper-kvasir upper-gi-tract
anatomical landmarks) is scored against one text prompt per class, and the
class with the highest image-text similarity wins.

Place the released checkpoint at `pretrained/endovlm_vitb16.pth` and run this
notebook from the repository root.

In [ ]:
import os
import sys
import glob

# Optional: HuggingFace mirror for model/text-encoder downloads.
# Set before importing open_clip so huggingface_hub picks it up.
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

import torch
import torch.nn.functional as F
import open_clip
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

PROJECT_ROOT = os.path.abspath(os.getcwd())  # run this notebook from the repo root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
from main_pretrain import get_args_parser
from endovlm import build_endovlm

args = get_args_parser().parse_args([
    '--model', 'vit_base_patch16',
    # skip the DINOv3 backbone pre-load; the EndoVLM checkpoint below carries the trained weights
    '--train_from_scratch',
])

CKPT_PATH = 'pretrained/endovlm_vitb16.pth'

model = build_endovlm(model_type=args.model, args=args)
msg = model.load_state_dict(torch.load(CKPT_PATH, map_location='cpu', weights_only=False)['model'], strict=False)
print(f"Missing keys: {len(msg.missing_keys)}, unexpected keys: {len(msg.unexpected_keys)}")
model = model.to(device).eval()

tokenizer = open_clip.get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
transform = transforms.Compose([
    transforms.Resize((args.input_size, args.input_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])


In [ ]:
SITE_PROMPTS = {
    "z-line": "An endoscopic image of z-line.",
    "retroflex-stomach": "An endoscopic image of fundus.",
    "pylorus": "An endoscopic image of pylorus.",
}
CLASS_NAMES = list(SITE_PROMPTS)

image_paths = sorted(glob.glob(os.path.join('data', 'zeroshot', 'hyper-kvasir_*.jpg')))

def gt_from_path(path):
    # hyper-kvasir_<class>.jpg -> <class>
    name = os.path.basename(path)[len('hyper-kvasir_'):]
    return os.path.splitext(name)[0]


In [ ]:
@torch.no_grad()
def get_image_features(img_tensor):
    latent, _, _ = model.forward_encoder(img_tensor.unsqueeze(0).to(device), 0.0)
    cls_tokens = latent[:, 0]
    patch_tokens = latent[:, 1 + model.encoder.n_storage_tokens:]
    feat = model.image_feat_projection(torch.cat([cls_tokens, patch_tokens.mean(dim=1)], dim=1))
    return F.normalize(model.image_projection_fg(feat), p=2, dim=-1)

@torch.no_grad()
def get_text_features(texts):
    return F.normalize(model.forward_text(tokenizer(texts).to(device)), p=2, dim=-1)


In [ ]:
text_features = get_text_features([SITE_PROMPTS[c] for c in CLASS_NAMES])

fig, axes = plt.subplots(1, len(image_paths), figsize=(3 * len(image_paths), 3.2))
for ax, path in zip(axes, image_paths):
    img = Image.open(path).convert('RGB')
    feats = get_image_features(transform(img))
    probs = torch.softmax((feats @ text_features.T) * model.logit_scale.exp(), dim=-1)[0].cpu()
    pred, gt = CLASS_NAMES[int(probs.argmax())], gt_from_path(path)
    ax.imshow(img)
    ax.set_title(f"gt: {gt}\npred: {pred} ({probs.max():.2f})", fontsize=9)
    ax.axis('off')
    print(f"{os.path.basename(path)}: gt={gt} pred={pred} | " +
          ", ".join(f"{c}={p:.2f}" for c, p in zip(CLASS_NAMES, probs)))
plt.tight_layout()
plt.show()
